# Logistic Regression Experiments
This notebook is a lightweight sandbox for primitive test runs.

Run cells in order from top to bottom.

In [22]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

# Data root in this repo
BASE_DIR = "datasets_full.csv/datasets_full.csv"
USERS_FILE = "users.csv"

DATASETS = {
    "genuine_accounts.csv": 0,
    "social_spambots_1.csv": 1,
    "social_spambots_2.csv": 1,
    "social_spambots_3.csv": 1,
    "traditional_spambots_1.csv": 1,
    "traditional_spambots_2.csv": 1,
    "traditional_spambots_3.csv": 1,
    "traditional_spambots_4.csv": 1,
}

USER_FEATURES = [
    "statuses_count", "followers_count", "friends_count",
    "favourites_count", "listed_count", "default_profile",
    "default_profile_image", "verified"
]

TWEET_FEATURES = [
   "text", "reply_count", "favorite_count", "num_urls", "num_mentions"
]

print("Dataset root:", BASE_DIR)
print("Files configured:", len(DATASETS))

Dataset root: datasets_full.csv/datasets_full.csv
Files configured: 8


In [23]:
def resolve_users_csv(base_dir, dataset_entry):
    """Resolve dataset entry to a users.csv path."""
    candidates = [
        os.path.join(base_dir, dataset_entry),
        os.path.join(base_dir, dataset_entry, USERS_FILE),
        os.path.join(base_dir, dataset_entry, dataset_entry, USERS_FILE),
    ]

    for p in candidates:
        if os.path.isfile(p):
            return p
    return None

def load_all_data(dataset_map, base_dir):
    """Load all configured datasets and attach binary labels."""
    frames = []
    missing_files = []

    for dataset_entry, label in dataset_map.items():
        path = resolve_users_csv(base_dir, dataset_entry)
        if path is None:
            missing_files.append(dataset_entry)
            continue

        df = pd.read_csv(
            path,
            encoding="utf-8",
            on_bad_lines="skip",
            low_memory=False,
        )
        df["label"] = int(label)
        df["source_file"] = dataset_entry
        frames.append(df)
        print(f"Loaded {dataset_entry}: {len(df):,} rows")

    if missing_files:
        print("\nMissing dataset entries:")
        for p in missing_files:
            print("-", p)

    if not frames:
        raise ValueError("No dataset files were loaded. Check BASE_DIR and DATASETS.")

    all_data = pd.concat(frames, ignore_index=True)
    print(f"\nTotal rows loaded: {len(all_data):,}")
    return all_data

def load_tweets(base_dir, dataset_entry):
    # possible paths where tweets.csv is
    candidates = [
        os.path.join(base_dir, dataset_entry, "tweets.csv"),
        os.path.join(base_dir, dataset_entry, dataset_entry, "tweets.csv"),
    ]
    
    for path in candidates:
        
        if os.path.isfile(path):
            available = ["user_id"] + TWEET_FEATURES
            
            chunks = []
            for chunk in pd.read_csv(path, usecols=lambda c: c in available, 
                                    chunksize=100_000, encoding="utf-8", on_bad_lines="skip"):
                chunks.append(chunk)
                
            dataframe = pd.concat(chunks, ignore_index=True)
                
            agg_cols = [c for c in TWEET_FEATURES if c in dataframe.columns]
            dataframe[agg_cols] = dataframe[agg_cols].apply(pd.to_numeric, errors="coerce")
            agg_cols = [c for c in agg_cols if pd.api.types.is_numeric_dtype(dataframe[c])]

            if not agg_cols:
                return None

            return dataframe.groupby("user_id")[agg_cols].mean().reset_index()
                        
    return None

# Block 1: Data loading (full configured datasets)
raw = load_all_data(DATASETS, BASE_DIR)

# load tweets into raw 
tweet_aggs = []
for dataset_entry in DATASETS:
    agg = load_tweets(BASE_DIR, dataset_entry)
    if agg is not None:
        tweet_aggs.append(agg)

if tweet_aggs:
    all_tweets = pd.concat(tweet_aggs, ignore_index=True)
    
    if "id" in raw.columns:
        raw = raw.rename(columns={"id": "user_id"})
    raw = raw.merge(all_tweets, on="user_id", how="left")

raw.shape

Loaded genuine_accounts.csv: 3,474 rows
Loaded social_spambots_1.csv: 991 rows
Loaded social_spambots_2.csv: 3,457 rows
Loaded social_spambots_3.csv: 464 rows
Loaded traditional_spambots_1.csv: 1,000 rows
Loaded traditional_spambots_2.csv: 100 rows
Loaded traditional_spambots_3.csv: 403 rows
Loaded traditional_spambots_4.csv: 1,128 rows

Total rows loaded: 11,017


(11017, 49)

In [ ]:
# Block 2: Slicing and preprocessing
available_numeric = [c for c in USER_FEATURES if c in raw.columns]
if not available_numeric:
    raise ValueError("No numeric feature columns found in raw data.")

X = raw[available_numeric].copy()
X = X.apply(pd.to_numeric, errors="coerce")

tweet_features = [c for c in TWEET_FEATURES if c in raw.columns]
X = pd.concat([X, raw[tweet_features]], axis=1)

X = X.fillna(0)

y = raw["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Class balance:\n", y.value_counts(dropna=False))

Train shape: (8813, 13)
Test shape: (2204, 13)
Class balance:
 label
1    7543
0    3474
Name: count, dtype: int64


In [29]:
# Block 3: Training
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_scaled, y_train)

print("Training complete.")
print("Model intercept:", model.intercept_[0])

Training complete.
Model intercept: 5.039737665645477


In [30]:
# Block 4: Evaluation
pred = model.predict(X_test_scaled)

print("Accuracy:", round(accuracy_score(y_test, pred), 4))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, pred))
print("\nClassification Report:")
print(classification_report(y_test, pred, digits=4))

coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0],
})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df = coef_df.sort_values("abs_coefficient", ascending=False)

print("\nTop 10 features by |coefficient|:")
display(coef_df.head(10))

Accuracy: 0.9496

Confusion Matrix:
[[ 662   33]
 [  78 1431]]

Classification Report:
              precision    recall  f1-score   support

           0     0.8946    0.9525    0.9226       695
           1     0.9775    0.9483    0.9627      1509

    accuracy                         0.9496      2204
   macro avg     0.9360    0.9504    0.9427      2204
weighted avg     0.9513    0.9496    0.9500      2204


Top 10 features by |coefficient|:


,feature,coefficient,abs_coefficient
3,favourites_count,-12.299907,12.299907
11,num_urls,7.397753,7.397753
12,num_mentions,7.214024,7.214024
10,favorite_count,6.474223,6.474223
2,friends_count,-1.773334,1.773334
4,listed_count,1.561926,1.561926
0,statuses_count,-0.673174,0.673174
5,default_profile,-0.630041,0.630041
7,verified,-0.595373,0.595373
1,followers_count,0.282636,0.282636


## Run Order
1. Cell 2: imports and constants.
2. Cell 3: full data loading.
3. Cell 4: slicing and preprocessing.
4. Cell 5: training.
5. Cell 6: evaluation.

## Notes
- This notebook now loads all configured dataset files directly.
- If runtime or memory gets heavy, reduce loaded columns or process in chunks.